# Chapter 10: Leader Election
*From: Database Internals*

## TL;DR

- A **leader** (coordinator) is a single process that drives a distributed algorithm, replacing peer-to-peer chatter with hub-and-spoke coordination — fewer messages, simpler ordering, one place to hold global state.
- Every basic leader-election algorithm in this chapter — **bully**, **next-in-line failover**, **candidate/ordinary**, **invitation**, **ring** — guarantees **liveness** (eventually some leader emerges) but *not* **safety** (at most one leader). Network partitions produce **split brain**.
- The algorithms trade along three axes: **message count** (O(N^2) bully vs O(N) ring vs O(C) candidate/ordinary), **topology assumptions** (ranks, ring, candidate sets), and **merge semantics** (bully overwrites, invitation merges).
- Leader election is *itself* a consensus problem. To get safety you need a cluster-wide **quorum** — which is what **Multi-Paxos**, **Raft**, and **ZAB** provide, with built-in leader election baked into the protocol.
- Production systems embrace the multiple-leader possibility and lean on **stable leader election** + failure detection + quorum-based conflict resolution rather than pretending safety comes from the election algorithm alone.

## Requirements

### Functional Requirements

- Elect exactly one leader at system start and after every leader crash (deterministic outcome).
- Every correct process must eventually agree on the new leader's identity (decision is effective for all participants).
- Detect leader failure and trigger a new election without manual intervention.
- Notify all peers of the leader's identity — unlike distributed locking, **identity matters** to clients.

### Non-Functional Requirements

- **Liveness**: the system must not remain in the "electing" state indefinitely.
- **Message efficiency**: election rounds are cheap enough that the amortized cost is invisible outside failure events.
- **Leader stability**: once elected, a leader should keep its role for a long time (long-lived leaders amortize coordination).
- **Partition behavior**: the algorithm must document how it behaves when the cluster splits — most of these admit split brain and leave safety to a higher layer.
- **Failure-detector coupling**: elections are triggered by a failure detector (Chapter 9); mis-tunings cascade into election thrash.

## Back-of-Envelope Estimation

Before picking an algorithm, size the worst-case message traffic and reelection latency for each family. The numbers below are the raw message counts around a single election round — the budget a reader should carry into the deep dives.

In [ ]:
# Message-count and latency comparison across leader-election algorithms.
# N   = cluster size
# C   = candidate subset (candidate/ordinary)
# rtt = one-way message hop (seconds)

from math import ceil

N   = 100
C   = 10      # candidate subset size for candidate/ordinary
rtt = 0.005   # 5 ms one-hop in a single datacenter

def bully_msgs(n):
    # Worst case: lowest-ranked node starts; each node contacts every higher-ranked peer
    # plus alive responses plus final broadcast. Dominant term is O(N^2).
    election  = sum(range(1, n))        # i-th ranked contacts (n-i) higher peers
    alive     = sum(range(1, n))        # higher peers respond
    elected   = n - 1                   # winner broadcasts to lower ranks
    return election + alive + elected

def bully_next_in_line_msgs(n, alternatives=3):
    # Detector contacts the highest-ranked alternative; it acks and broadcasts.
    return 1 + 1 + (n - 1)

def candidate_ordinary_msgs(n, c):
    # Ordinary contacts c candidates, they reply, winner is announced to n-1 others.
    return c + c + (n - 1)

def ring_msgs(n):
    # Full ring traversal to collect live set + second traversal to announce winner.
    return 2 * n

def invitation_msgs_merge(n):
    # Pairwise merges: log2(n) merge rounds, each round touching the smaller group.
    # Small-group-notified heuristic keeps per-round cost to n/2 in the worst case.
    rounds = ceil((n.bit_length()))
    return rounds * (n // 2)

algos = [
    ("Bully (worst case)",           bully_msgs(N),                  3 * rtt),
    ("Bully + next-in-line",         bully_next_in_line_msgs(N),     2 * rtt),
    ("Candidate/Ordinary",           candidate_ordinary_msgs(N, C),  3 * rtt),
    ("Ring",                         ring_msgs(N),                   N * rtt),
    ("Invitation (growth to 1 grp)", invitation_msgs_merge(N),       N.bit_length() * rtt),
]

print(f"Cluster size N={N}, candidate C={C}, rtt={rtt*1000:.0f} ms\n")
print(f"{'Algorithm':30s} {'Messages':>12s} {'Reelection latency':>22s}")
print("-" * 68)
for name, msgs, lat in algos:
    print(f"{name:30s} {msgs:>12,d} {lat*1000:>18.1f} ms")

**Reading the numbers.**

- **Bully** is quadratic: at N=100 it is ~10 000 messages on the worst path; acceptable only because elections are rare.
- **Next-in-line** collapses a bully election to ~N messages in the common case by pre-publishing a failover list.
- **Candidate/ordinary** caps cost at O(C) where C is usually small and fixed — message count barely grows with N.
- **Ring** is linear in *messages* but latency is proportional to N hops; great on bandwidth, bad on reelection latency.
- **Invitation** spends O(N log N) merge messages spread across groups — useful when you do not want a from-scratch election on every failure.

## High-Level Design

Every algorithm in this chapter follows the same shape: a triggering event is observed, an election protocol runs, and the result is disseminated to clients and peers. Only the protocol body differs.

```mermaid
graph LR
  FD[Failure Detector<br/>Ch. 9] -->|"leader suspected"| TRIG[Election Trigger]
  TRIG --> PROTO{Election Protocol}
  PROTO -->|bully / ring / ...| WINNER[Leader Chosen]
  WINNER --> ANN[Announcement<br/>broadcast to peers]
  ANN --> APP[Application Layer<br/>total order, replication,<br/>metadata updates]
  APP -.split brain.-> PROTO
```

The dotted feedback edge is the uncomfortable truth: when the chosen protocol admits split brain, the "application layer" has to detect and resolve two leaders after the fact — that is what consensus algorithms end up doing.

## Deep Dive 1: Leader Election Fundamentals

A leader absorbs coordination cost. Instead of every pair of processes reaching agreement on each step, processes route requests through the leader, which orders them and disseminates the result. The classic application is **total-order broadcast**: the leader stamps an order on incoming messages and replicates them.

```mermaid
graph LR
  subgraph Peer-to-peer
    A1[P1] --- A2[P2]
    A1 --- A3[P3]
    A1 --- A4[P4]
    A2 --- A3
    A2 --- A4
    A3 --- A4
  end
  subgraph "Leader-driven"
    L[Leader] --> B2[P2]
    L --> B3[P3]
    L --> B4[P4]
  end
```

**What leader election is not.** It resembles distributed locking but differs in three ways:

| Property | Distributed Lock | Leader Election |
|---|---|---|
| Identity visible to peers | No — only liveness matters | **Yes** — peers route to the leader |
| Preference allowed | No — preference starves peers | **Yes** — ranks and priorities are common |
| Holder tenure | Short, must eventually release | **Long-lived** — leader keeps role until it crashes |

**Bottleneck mitigation.** A single system-wide leader becomes a scaling ceiling. The standard cure is **partitioning**: carve the data into non-intersecting replica sets, and give each set its own leader. Spanner does exactly this.

## Deep Dive 2: Bully Algorithm (and Next-In-Line Failover)

The **bully algorithm** (Garcia-Molina) assumes each process has a unique rank. The highest-ranked live process wins. Any process that notices the leader is gone starts a round in three steps: **Election -> Alive -> Elected**.

```mermaid
sequenceDiagram
  participant P3
  participant P4
  participant P5
  participant P6
  Note over P6: Previous leader P6 has crashed.
  P3->>P4: Election
  P3->>P5: Election
  P3->>P6: Election (no response)
  P4-->>P3: Alive
  P5-->>P3: Alive
  P3->>P5: You are highest — take over
  P5->>P3: Elected (you are under me)
  P5->>P4: Elected
  P5->>P2: Elected
  P5->>P1: Elected
```

**Why it is attractive.** Each step is obvious, the invariant ("highest rank wins") is easy to reason about, and the messages are simple.

**Where it breaks.**
1. **Split brain under partitions** — each partition can elect its own top-ranked live node.
2. **Reelection thrash** — if the top-ranked node is flapping, it wins, dies, wins again. Mitigate by folding host-quality metrics into the rank or adding hysteresis.
3. **Quadratic worst case** — O(N^2) messages when the lowest-ranked node triggers the election.

**Next-in-line failover.** The elected leader publishes an ordered list of alternative leaders. On detection, the observer contacts the highest-ranked alternative directly; if it is alive, it broadcasts "I am the new leader" without a full bully round.

```mermaid
sequenceDiagram
  participant P3
  participant P4
  participant P5
  participant P6
  Note over P6: Leader P6 (alt list = {5, 4}) crashes.
  P3->>P5: You are next in line — alive?
  P5-->>P3: Alive
  P5->>P1: Elected
  P5->>P2: Elected
  P5->>P3: Elected
  P5->>P4: Elected
```

Benefit: three messages in the common case instead of O(N^2). Cost: the previous leader must have published a fresh alt list, and an observer that is itself the top alternative short-circuits further.

## Deep Dive 3: Candidate / Ordinary Optimization

This variant separates the population into two sets:

- **Candidates** — eligible to become leader.
- **Ordinaries** — only participate as voters / observers.

Only candidates reply with "Alive". An ordinary detecting failure polls *all* candidates, picks the highest-ranked responder, and announces. Message count drops from O(N^2) to O(C) where C is the (small) candidate set.

```mermaid
sequenceDiagram
  participant P4 as P4 (ordinary)
  participant P1 as P1 (candidate)
  participant P2 as P2 (candidate)
  participant P3 as P3 (candidate)
  participant P6 as P6 (old leader, crashed)
  P4->>P1: Election
  P4->>P2: Election
  P4->>P3: Election
  P1-->>P4: Alive
  P2-->>P4: Alive
  P3-->>P4: Alive
  Note over P4: Highest-ranked candidate alive = P2.
  P4->>ALL: Elected(P2)
```

**The simultaneous-election problem.** Multiple ordinaries can notice the failure at the same time and start competing rounds. The algorithm adds a per-process delay **delta** — a tiebreaker chosen per node and larger than a typical RTT. Higher-priority (better-connected, more stable) nodes get smaller delta and initiate first, suppressing duplicate rounds.

**Trade-offs.** You gain O(C) messages and natural throttling via delta. You lose the self-healing property that *any* node could become leader, and you take on the burden of tuning delta per node.

## Deep Dive 4: Invitation Algorithm

The invitation algorithm inverts the question: instead of outranking each other, processes **build groups by invitation**. Each process starts as a singleton group where it is the leader. Leaders invite outsiders; when a leader invites another leader, the two groups **merge**.

```mermaid
graph LR
  subgraph "Step a — four singleton groups"
    A1[P1*] ---|invites| A2[P2]
    A3[P3*] ---|invites| A4[P4]
  end
  subgraph "Step b — two merged groups"
    B1[P1*] --- B2[P2]
    B3[P3*] --- B4[P4]
    B1 ---|merge| B3
  end
  subgraph "Step c — single group"
    C1[P1*] --- C2[P2]
    C1 --- C3[P3]
    C1 --- C4[P4]
  end
```

(`*` = group leader.) The **merge efficiency heuristic**: when two groups combine, the leader of the *larger* group becomes the leader of the merged group, because only the smaller group's members need to be notified of the new leader.

**What is unusual.** The algorithm **allows multiple leaders by definition** — before every merge there are several. That means you do not need to restart the election on every failure; you just let groups fracture and re-merge as links recover.

**Where it shines.** Systems that are already partition-tolerant and want membership churn to be cheap. Where it fails: anything that requires a single cluster-wide order immediately.

## Deep Dive 5: Ring Algorithm

All nodes form a logical ring. Each knows its successor. On detecting leader failure, a node sends an **Election** message with a path (or running max) around the ring, skipping unreachable successors. When the message returns, the highest-ranked live node is chosen and an **Elected** message circulates.

```mermaid
sequenceDiagram
  participant P3
  participant P4
  participant P5
  participant P1
  participant P2
  Note over P3: P3 starts with empty path
  P3->>P4: Election path=3
  P4->>P5: Election path=3,4
  P5->>P1: Election path=3,4,5 (skips crashed P6)
  P1->>P2: Election path=3,4,5,1
  P2->>P3: Election path=3,4,5,1,2
  Note over P3: Max of path is 5 -- start announcement lap
  P3->>P4: Elected P5
  P4->>P5: Elected P5
  P5->>P1: Elected P5
  P1->>P2: Elected P5
  P2->>P3: Elected P5
```

**Space optimization.** Because `max` is commutative, the payload does not need to carry the full live set — a single running maximum suffices. This is the same trick as CRDT counters.

**Where it breaks.** A ring is trivially **partitionable**: cut two edges and you have two rings, each electing its own leader. Safety is not preserved. Also, *latency* scales with N hops; a large ring has slow elections even when messages are cheap.

## Deep Dive 6: Split Brain, Stable Leader Election, and Consensus

Every algorithm so far shares one property: under a network partition, each partition can elect its own leader. Call this **split brain**. Two leaders serving the same role is, in general, catastrophic — writes diverge, logs fork, clients see inconsistent state.

```mermaid
graph TB
  subgraph "Before partition"
    L0[Leader P5] --- N1[P1]
    L0 --- N2[P2]
    L0 --- N3[P3]
    L0 --- N4[P4]
  end
  subgraph "After partition — two leaders"
    L1[Leader P5<br/>left half] --- M1[P1]
    L1 --- M2[P2]
    L2[Leader P3<br/>right half] --- M3[P4]
    L1 -.X.- L2
  end
```

**The fix: quorum.** Require any leader to hold a **cluster-wide majority** of votes. Under any partition, at most one side has a majority, so at most one side can have a leader. This is the core of:

| Protocol | How it elects a leader | Safety mechanism |
|---|---|---|
| **Multi-Paxos** | Proposer becomes stable leader after completing phase 1 with a quorum | Conflict between two proposers is resolved in phase 2 — only one value gets a quorum of accepts |
| **Raft** | Candidates request votes; winner with majority becomes leader for a **term** | Terms are monotonically increasing; stale-term leaders step down on contact with a higher term |
| **ZAB** | Epoch-numbered elections with FLE (Fast Leader Election) selecting the peer with the latest log | Followers only accept proposals from the leader of the current epoch |

**Stable leader election.** Aguilera's stable leader election combines rounds with a unique leader per round + timeout-based failure detection. Once elected, the leader retains its position as long as it remains reachable. This is the bridge between the naive algorithms in this chapter and the consensus protocols in Chapter 14.

**Liveness vs safety trade-off in production.** Many systems deliberately allow *transient* multiple leaders to preserve availability, then resolve conflicts at the quorum layer:

- Multi-Paxos lets two proposers both enter phase 2; only one collects a quorum of accepts.
- Raft lets a stale leader keep serving reads until it notices a higher term and steps down.

**Election is consensus.** If you can reach consensus about who the leader is, you can reach consensus about anything — *electing the leader is isomorphic to agreeing on a single value*. That is why production databases and coordination services (ZooKeeper, etcd, Consul) do not implement the naive algorithms in this chapter — they implement full consensus and extract the leader as a by-product.

## Trade-offs and Alternatives

| Approach | Messages (worst case) | Latency | Safety under partition | Assumptions | Best for |
|---|---|---|---|---|---|
| **Bully** | O(N^2) | ~3 RTT | Split brain | Total order on process IDs | Small clusters; pedagogy |
| **Next-in-line** | O(N) common, O(N^2) fallback | ~2 RTT | Split brain | Leader publishes alt list | Fast failover with simple rank model |
| **Candidate/Ordinary** | O(C) where C << N | ~3 RTT | Split brain | Candidate subset + delta tuning | Large clusters where only a few nodes are leader-worthy |
| **Invitation** | O(N log N) amortized across merges | O(log N) RTT | Multiple leaders by design | Groups can merge pairwise | Partition-tolerant membership churn |
| **Ring** | O(N) messages, N-hop latency | O(N) RTT | Split brain on ring cuts | Logical ring topology maintained | Bandwidth-constrained clusters; token-passing patterns |
| **Stable / Consensus-based** (Raft, Paxos, ZAB) | O(N) per round, quorum reads | ~2 RTT | **Safe** — at most one leader per term | Majority available | Production coordination, metadata, config |

**How to pick.** If you only need liveness and a hint of who is in charge (membership gossip, load-balancer shard hints) any of the naive algorithms works. If incorrect behavior under split brain is not tolerable — config service, transaction coordinator, primary replica — use a consensus protocol. Do not reinvent stable leader election on top of bully and hope partitions never happen.

## Key Takeaways

1. A leader collapses peer-to-peer coordination into hub-and-spoke, dramatically reducing message count at the cost of creating a single failure point and a bottleneck — **partition data per leader** to escape the bottleneck.
2. All basic leader-election algorithms guarantee **liveness but not safety**. Under partitions, each side elects its own leader. Split brain is the rule, not the exception.
3. **Bully** is the canonical baseline — highest rank wins, O(N^2) messages. **Next-in-line** cuts the common case to O(N) by pre-publishing failover lists.
4. **Candidate/Ordinary** decouples the leader pool from the cluster size, turning election cost into O(C); the **delta** tiebreaker suppresses concurrent rounds without explicit coordination.
5. **Invitation** embraces multiple leaders, growing and merging groups by invitation — it trades safety for cheap membership churn and avoids from-scratch reelections.
6. **Ring** traversal achieves O(N) messages with O(N) latency and (because `max` is commutative) can circulate a running maximum instead of the full live set.
7. **Electing a leader is consensus.** Production systems embed leader election inside Multi-Paxos, Raft, or ZAB, using cluster-wide quorums to guarantee *at most one* leader per term — and use the naive algorithms in this chapter only for non-safety-critical coordination.

## See Also

- [[01-leader-election-fundamentals]]
- [[02-bully-algorithm]]
- [[03-candidate-ordinary-optimization]]
- [[04-invitation-algorithm]]
- [[05-ring-algorithm]]
- [[06-leader-election-and-consensus]]